# clef-flash: avaliação e fine-tuning no Colab

O código vem do GitHub (você edita no PC, dá push e roda aqui). Resultados e checkpoints vão para o Google Drive em `MyDrive/system-one/`; no PC, `python sync_mlflow.py` traz os runs para o seu MLflow.

**Antes de começar**
1. Ambiente de execução → Alterar tipo → **G4** (ou H100/A100).
2. No Drive, crie `MyDrive/system-one/data/en/` e envie `train.jsonl`, `val.jsonl`, `test.jsonl` e `data_stats.json` de `D:\laya-treino\data\en` (o mesmo teste do Laya).
3. Opcional: em Secrets (ícone de chave), crie `HF_TOKEN` com um token de leitura do Hugging Face (o modelo é público; o token só evita limite de download).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
REPO = 'https://github.com/Lucasrrafael/system-one-model-fine-tuning.git'
BRANCH = 'clef-flash'
DRIVE = '/content/drive/MyDrive/system-one'
G4_UNIDADES_POR_HORA = 8.7  # medição da comunidade; confira no painel do Colab

## 1. Código e dependências

In [ ]:
import os
if not os.path.exists('/content/repo'):
    !git clone -b {BRANCH} {REPO} /content/repo
else:
    !git -C /content/repo fetch -q && git -C /content/repo checkout -q {BRANCH} && git -C /content/repo pull -q
%cd /content/repo
!git log -1 --oneline

In [ ]:
!pip install -q -r requirements-colab.txt
# O Colab vem com um torchao antigo (0.10) que faz o peft falhar ao criar o LoRA.
# O projeto não usa torchao, então ele sai.
!pip uninstall -q -y torchao
import importlib.util
print('flash-linear-attention:', 'ok' if importlib.util.find_spec('fla') else 'FALTANDO: a atenção linear vai rodar no modo lento')
# Opcional: causal-conv1d compila CUDA e pode levar 10+ min; o ganho é pequeno perto do fla.
INSTALAR_CAUSAL_CONV1D = False
if INSTALAR_CAUSAL_CONV1D:
    !pip install -q causal-conv1d --no-build-isolation || echo 'causal-conv1d não instalou; segue sem'

In [ ]:
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN carregado dos Secrets')
except Exception:
    print('Sem HF_TOKEN nos Secrets (opcional)')

## 2. Dados (os mesmos arquivos do PC, conferidos pelo hash)

In [ ]:
import hashlib, json, shutil
src, dst = f'{DRIVE}/data/en', '/content/work/data/en'
os.makedirs(dst, exist_ok=True)
for name in ['train.jsonl', 'val.jsonl', 'test.jsonl', 'data_stats.json']:
    shutil.copy(f'{src}/{name}', dst)
stats = json.load(open(f'{dst}/data_stats.json'))
for split in ['train', 'val', 'test']:
    digest = hashlib.sha256(open(f'{dst}/{split}.jsonl', 'rb').read()).hexdigest()
    assert digest == stats[split]['sha256'], f'{split}: hash diferente do gerado no PC'
    print(f"{split}: {stats[split]['rows']} linhas, hash ok")

## 3. Sondagem (200 tickets, 1 época)

Mede segundos por passo e VRAM antes de gastar horas. O modelo baixado fica no disco do Colab para o run completo.

In [ ]:
cut = '/content/work/data/en-train200-eval100'
os.makedirs(cut, exist_ok=True)
for split, n in [('train', 200), ('val', 100), ('test', 100)]:
    with open(f'{dst}/{split}.jsonl') as f_in, open(f'{cut}/{split}.jsonl', 'w') as f_out:
        f_out.writelines(f_in.readlines()[:n])
!ENV_FILE=configs/clef-colab.env PYTHONUNBUFFERED=1 MAX_TRAIN_SAMPLES=200 MAX_EVAL_SAMPLES=100 EPOCHS=1 LOG_EVERY=5 MLFLOW_EXPERIMENT=clef-sondagem DELETE_MODEL_AFTER_RUN=false python run.py

In [ ]:
import math
SEG_POR_PASSO = 0.0  # copie o 's/passo' da última linha de progresso da sondagem
itens = stats['train']['rows'] - 400  # 400 vão para a calibração
passos = math.ceil(itens / 8) * 2     # MICRO_BATCH=8, EPOCHS=2
horas = passos * SEG_POR_PASSO / 3600
print(f'treino: ~{horas:.1f} h  ->  ~{horas * G4_UNIDADES_POR_HORA:.0f} unidades de computação (sem contar avaliação e download)')

## 4. Run completo

Avalia o clef sem treino no teste, treina LoRA + cabeça por 2 épocas, calibra e avalia de novo. A cada época o run e o MLflow são copiados para o Drive.

In [ ]:
!ENV_FILE=configs/clef-colab.env PYTHONUNBUFFERED=1 python run.py

## Se o Colab desconectar

1. Rode de novo as seções 1 e 2.
2. Na célula abaixo, ponha o nome do run (pasta em `MyDrive/system-one/runs/`) e rode. Ele restaura o banco do MLflow do Drive e continua do `checkpoint_latest`, sem refazer a avaliação base.

In [ ]:
RUN = ''  # ex.: clef-flash-lora-20261101-153000
os.makedirs('/content/mlflow', exist_ok=True)
if os.path.exists(f'{DRIVE}/mlflow/mlflow.db'):
    shutil.copy(f'{DRIVE}/mlflow/mlflow.db', '/content/mlflow/mlflow.db')
    shutil.copytree(f'{DRIVE}/mlflow/artifacts', '/content/mlflow/artifacts', dirs_exist_ok=True)
ckpt = f'{DRIVE}/runs/{RUN}/checkpoint/checkpoint_latest'
assert RUN and os.path.exists(ckpt), 'preencha RUN com uma pasta que tenha checkpoint_latest'
!ENV_FILE=configs/clef-colab.env PYTHONUNBUFFERED=1 RESUME_FROM={ckpt} EVAL_BASELINE=false MLFLOW_RUN_NAME={RUN}-retomado python run.py

## 5. Levar para o seu MLflow

Baixe a pasta `MyDrive/system-one/mlflow` (o Drive gera um zip), extraia em `D:\laya-treino\colab-mlflow` e, na pasta do projeto:

```powershell
D:\laya-treino\.venv\Scripts\python.exe sync_mlflow.py --source D:\laya-treino\colab-mlflow\mlflow
```

Os runs completos aparecem em http://localhost:5000 no experimento `system-one-model-priority`, junto com os do Laya; as sondagens ficam em `clef-sondagem`.